In [1]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("datasets/dataset_final.csv")

In [3]:
df.columns

Index(['Count_www', 'Path_Length', 'Count_/', 'Domain_URL_Ratio', 'URL/Path',
       'Character_Repetition', 'Having_Path', 'fd_length', 'Count_Digit',
       'FractalDimension', 'Digit/Letter', 'Kolmogorov_Complexity',
       'Domain_Length_Of_URL', 'Longest_Word', 'Count_Letter',
       'ShannonEntropy', 'Host_Precense_Of_Digit', 'Tld_Length', 'Label'],
      dtype='object')

In [4]:
X = df.drop(columns=["Label"])
y = df["Label"]

In [10]:
import pandas as pd
from sklearn.base import clone

# ... (Modeller ve X_train/test tanımları yukarıda olduğu gibi kalıyor) ...

results = []
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

for name, model in models.items():
    try:
        print(f">>> {name} eğitiliyor...")
        current_model = clone(model)
        current_model.fit(X_train, y_train)
        y_pred = current_model.predict(X_test)
        
        row = {
            "Model": name,
            "Test_Accuracy": accuracy_score(y_test, y_pred),
            "Test_Precision": precision_score(y_test, y_pred),
            "Test_Recall": recall_score(y_test, y_pred),
            "Test_F1": f1_score(y_test, y_pred)
        }

        try:
            if hasattr(current_model, "predict_proba"):
                y_prob = current_model.predict_proba(X_test)[:, 1]
            else:
                y_prob = current_model.decision_function(X_test)
            row["Test_ROC_AUC"] = roc_auc_score(y_test, y_prob)
        except:
            row["Test_ROC_AUC"] = None

        print(f"    {name} için Cross-Validation yapılıyor...")
        cv_scores = cross_validate(clone(model), X, y, cv=5, scoring=scoring, n_jobs=-1)

        row["CV_Accuracy"] = cv_scores["test_accuracy"].mean()
        row["CV_F1"] = cv_scores["test_f1"].mean()
        row["CV_ROC_AUC"] = cv_scores["test_roc_auc"].mean()

        results.append(row)
        print(f"--- {name} tamamlandı! ---")
        
    except Exception as e:
        print(f"!!! {name} hatası: {e}")

# --- TABLO DÜZENLEME VE INDEX SIFIRLAMA ---
results_table = pd.DataFrame(results).sort_values("Test_Accuracy", ascending=False)

# Karışık indexleri (0, 5, 4...) siler ve 1, 2, 3... diye sıralar
results_table = results_table.reset_index(drop=True)
results_table.index = results_table.index + 1 

print("\n=== PERFORMANS KARŞILAŞTIRMA TABLOSU (Sıralı) ===")
print(results_table)

>>> Random Forest eğitiliyor...
    Random Forest için Cross-Validation yapılıyor...
--- Random Forest tamamlandı! ---
>>> Logistic Regression eğitiliyor...
    Logistic Regression için Cross-Validation yapılıyor...
--- Logistic Regression tamamlandı! ---
>>> Linear SVM eğitiliyor...
    Linear SVM için Cross-Validation yapılıyor...
--- Linear SVM tamamlandı! ---
>>> Naive Bayes eğitiliyor...
    Naive Bayes için Cross-Validation yapılıyor...
--- Naive Bayes tamamlandı! ---
>>> Decision Tree eğitiliyor...
    Decision Tree için Cross-Validation yapılıyor...
--- Decision Tree tamamlandı! ---
>>> XGBoost eğitiliyor...
    XGBoost için Cross-Validation yapılıyor...
!!! XGBoost hatası: 'super' object has no attribute '__sklearn_tags__'
>>> Gradient Boost eğitiliyor...
    Gradient Boost için Cross-Validation yapılıyor...
--- Gradient Boost tamamlandı! ---
>>> KNN eğitiliyor...
    KNN için Cross-Validation yapılıyor...
--- KNN tamamlandı! ---

=== PERFORMANS KARŞILAŞTIRMA TABLOSU (Sıralı) 